### Feature Engineering Summary

- Removed cancelled and diverted flights
- Defined binary delay target using FAA standard (>15 minutes)
- Eliminated post-flight and leakage features
- Dropped DIVERTED column after row filtering
- Engineered scheduled time and temporal features
- Integrated origin airport geographic metadata
- Safely handled missing airport information
- Created stratified train-test split


### Imports

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", 100)
RANDOM_STATE = 42

# ── Update these paths to match your data location ──
FLIGHTS_PATH  = "data/raw/flight_data.csv"
AIRPORTS_PATH = "data/raw/airports.csv"


### Load Raw Data

In [2]:
flights  = pd.read_csv(FLIGHTS_PATH)
airports = pd.read_csv(AIRPORTS_PATH)

print("Raw flights shape:", flights.shape)


Raw flights shape: (539747, 34)


### Remove Cancelled & Diverted Flights

In [3]:
flights = flights[
    (flights["CANCELLED"] == 0) &
    (flights["DIVERTED"]  == 0)
].copy()

print("After removing cancelled/diverted:", flights.shape)


After removing cancelled/diverted: (522269, 34)


### Define Target Variable

In [4]:
flights["DELAYED"] = (flights["ARR_DELAY_NEW"] > 15).astype(int)
print(flights["DELAYED"].value_counts(normalize=True))


DELAYED
0    0.818438
1    0.181562
Name: proportion, dtype: float64


### Drop Leakage & Post-Flight Columns

Also drops CANCELLED and DIVERTED columns — they were used only for row filtering above.

In [5]:
leakage_cols = [
    # post-flight outcomes
    "ARR_DELAY_NEW",
    "ARR_TIME",
    "ARR_TIME_BLK",
    "DEP_DELAY",
    "DEP_DELAY_NEW",
    "DEP_TIME",
    "TAXI_OUT",
    "TAXI_IN",
    "AIR_TIME",
    "CARRIER_DELAY",
    "WEATHER_DELAY",
    "NAS_DELAY",
    "SECURITY_DELAY",
    "LATE_AIRCRAFT_DELAY",
    "DIV_ACTUAL_ELAPSED_TIME",
    "CANCELLATION_CODE",
    # row-filter flags no longer needed as features
    "CANCELLED",
    "DIVERTED",
    # redundant ID
    "OP_CARRIER_AIRLINE_ID",
]

flights = flights.drop(columns=[c for c in leakage_cols if c in flights.columns])
print("After dropping leakage columns:", flights.shape)


After dropping leakage columns: (522269, 16)


### Time-Based Features (Scheduled Only)

In [6]:
flights["CRS_DEP_HOUR"] = flights["CRS_DEP_TIME"] // 100
flights["CRS_ARR_HOUR"] = flights["CRS_ARR_TIME"] // 100
flights = flights.drop(columns=["CRS_DEP_TIME", "CRS_ARR_TIME"])


### Temporal Flags

In [7]:
flights["IS_WEEKEND"] = flights["DAY_OF_WEEK"].isin([6, 7]).astype(int)


### Integrate Airport Metadata (Origin)

In [8]:
airports_small = airports[["iata_code", "lat_decimal", "lon_decimal", "altitude"]]

flights = flights.merge(
    airports_small,
    left_on="ORIGIN",
    right_on="iata_code",
    how="left"
).drop(columns=["iata_code"])

print("After airport merge:", flights.shape)


After airport merge: (524026, 20)


### Handle Missing Airport Metadata

In [9]:
flights = flights.dropna(subset=["lat_decimal", "lon_decimal", "altitude"])
print("After dropping missing airport metadata:", flights.shape)


After dropping missing airport metadata: (492756, 20)


### Final Sanity Check

In [10]:
print("Remaining nulls (top 10):")
print(flights.isnull().sum().sort_values(ascending=False).head(10))
print("\nFinal columns:", flights.columns.tolist())


Remaining nulls (top 10):
YEAR            0
MONTH           0
lon_decimal     0
lat_decimal     0
IS_WEEKEND      0
CRS_ARR_HOUR    0
CRS_DEP_HOUR    0
DELAYED         0
DISTANCE        0
DEP_TIME_BLK    0
dtype: int64

Final columns: ['YEAR', 'MONTH', 'DAY_OF_MONTH', 'DAY_OF_WEEK', 'FL_DATE', 'OP_UNIQUE_CARRIER', 'OP_CARRIER', 'ORIGIN', 'ORIGIN_CITY_NAME', 'DEST', 'DEST_CITY_NAME', 'DEP_TIME_BLK', 'DISTANCE', 'DELAYED', 'CRS_DEP_HOUR', 'CRS_ARR_HOUR', 'IS_WEEKEND', 'lat_decimal', 'lon_decimal', 'altitude']


### Feature / Target Split

In [11]:
X = flights.drop(columns=["DELAYED"])
y = flights["DELAYED"]


### Train-Test Split

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)


Train shape: (394204, 19)
Test shape : (98552, 19)


### Save Processed Data

In [13]:
output_dir = Path("data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

X_train.to_csv(output_dir / "X_train.csv", index=False)
X_test.to_csv(output_dir  / "X_test.csv",  index=False)
y_train.to_csv(output_dir / "y_train.csv", index=False)
y_test.to_csv(output_dir  / "y_test.csv",  index=False)

print("Processed data saved to data/processed/")


Processed data saved to data/processed/
